# Phase 1 Single-Slice — dynamicLRP × SigLIP-2 SMOKE TEST (01_single_slice)

> **Honest scope.** This notebook is an **end-to-end smoke test**, not the
> original overlay + three-sanity-controls finish line. dynamicLRP's
> op-coverage **does not cover SigLIP-2-so400m**: the `split_with_sizes` op in
> SigLIP-2's MAP-pool / attention head is outside the current engine, so
> `attribute()` raises a `RuntimeError` and **no LRP relevance / heatmap is
> produced for SigLIP-2**. This was reviewed at the 01-03 human-verify
> checkpoint and the user **explicitly declined the entire Fallback Ladder**
> (no custom Promise, no pre-pool/`use_attn_lrp`, no LXT, no captum IG):
> *"Smoke-test was good, it didn't crash. Let's leave well enough alone and
> move on."* The project is reframed as a **multi-model comparison of
> dynamic-LRP** — SigLIP-2 is one model and this is an **accepted per-model
> FINDING**. The Phase 1 D-02/D-03 visual-eyeball gate is **consciously
> WAIVED for SigLIP-2** (documented in `01-03-SUMMARY.md`, not a silent pass).

What this notebook DOES, end to end, exiting 0:
1. loads SigLIP-2 from the GCS weights mirror,
2. builds the `requires_grad` `(1,3,384,384)` pixel tensor for the **locked
   slice** (`manifest[-1]`, D-05) + locked control query `"a river"` (D-06),
3. runs the forward and **measures peak forward VRAM** (closes the STATE.md
   peak-VRAM blocker),
4. runs the dynamicLRP **coverage probe** and prints the op count,
5. calls `attribute()` in a `try/except` that **catches** the `RuntimeError`
   and prints the FINDING (not a traceback),
6. shows the **source map** inline so the slice is visually identified.

In [1]:
# Cell 1 - path wiring, imports, deterministic seed, CUDA fail-fast
import sys, os, random
from pathlib import Path

_REPO_ROOT = Path.cwd()
if (_REPO_ROOT / "notebooks").exists() is False and (_REPO_ROOT.name == "notebooks"):
    _REPO_ROOT = _REPO_ROOT.parent
for _p in (_REPO_ROOT / "third_party" / "dynamicLRP" / "src", _REPO_ROOT / "src"):
    _s = str(_p)
    if _s not in sys.path:
        sys.path.insert(0, _s)

import numpy as np
import torch
import transformers
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), (
    "CUDA required: SigLIP-2 + dynamic-LRP on so400m is impractical on CPU "
    "(CLAUDE.md). Run this notebook on the GPU VM."
)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("device      :", torch.cuda.get_device_name(0))
print("seed        :", SEED, "(torch/numpy/cuda set + recorded)")

/home/drdreadknee/mapclass/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch       : 2.7.1+cu126
transformers: 4.52.3
device      : NVIDIA L4
seed        : 0 (torch/numpy/cuda set + recorded)


In [2]:
# Cell 2 - resolve the LOCKED slice = manifest[-1] (D-05, highest LIST index)
from mapclass.manifest import load_manifest, entry_by_index

m = load_manifest()
entry = entry_by_index(m, -1)
print("manifest length :", len(m))
print("locked slice id :", entry["id"])
assert entry["id"] == "RUMSEY~8~1~344476~90112460", entry["id"]
print("D-05 satisfied  : highest list index, no richness sort, no user input")

manifest length : 1544
locked slice id : RUMSEY~8~1~344476~90112460
D-05 satisfied  : highest list index, no richness sort, no user input


In [3]:
# Cell 3 - load the GCS-mirrored singleton SigLIP-2 + processor; coverage probe
from mapclass.model_loader import get_model_and_processor
from mapclass.data_loader import load_slice
from mapclass.attribution import attribute, coverage_probe

model, processor = get_model_and_processor()
print("model class :", type(model).__name__)
print("vision tower:", type(model.vision_model).__name__)

# dynamicLRP static op-coverage probe BEFORE the relevance pass. This is the
# coverage artifact for the multi-model comparison (an artifact, not a gate).
_it, _ii, _am, _pil = load_slice(entry, "a river")
_ops = coverage_probe(model, _it, _ii, _am)
try:
    _names, _count, _graph = _ops
except (TypeError, ValueError):
    _count = len(_ops) if hasattr(_ops, "__len__") else _ops
print("dynamicLRP get_model_operations op count:", _count)
print(
    "NOTE: SigLIP-2's MAP-pool/attention head uses `split_with_sizes`, which "
    "is OUTSIDE the current dynamicLRP engine coverage (see the FINDING cell)."
)
del _it, _ii, _am, _pil
torch.cuda.empty_cache()

/home/drdreadknee/mapclass/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Cell 4 - locked slice + "a river": forward, PEAK FORWARD VRAM, source map.
# The forward succeeds; only the dynamicLRP relevance pass fails (Cell 5).
# This cell CLOSES the STATE.md peak-VRAM blocker by measuring the peak VRAM
# of the SigLIP-2 forward on the locked (1,3,384,384) slice tensor.
img_tensor, input_ids, attention_mask, pil = load_slice(entry, "a river")
print("pixel tensor shape :", tuple(img_tensor.shape),
      "| requires_grad =", bool(img_tensor.requires_grad))

torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _fwd = model(pixel_values=img_tensor, input_ids=input_ids,
                 attention_mask=attention_mask)
    _sim = float(_fwd.logits_per_image[0, 0].item())
PEAK_FWD_VRAM_BYTES = int(torch.cuda.max_memory_allocated())
PEAK_FWD_VRAM_GB = PEAK_FWD_VRAM_BYTES / (1024 ** 3)
del _fwd
torch.cuda.empty_cache()

print(f'similarity logits_per_image[0,0] ("a river") : {_sim:.4f}')
print(f"PEAK FORWARD VRAM (single so400m forward)    : "
      f"{PEAK_FWD_VRAM_GB:.3f} GB ({PEAK_FWD_VRAM_BYTES:,} bytes)")
print("STATE.md peak-VRAM blocker: CLOSED (forward peak measured + recorded "
      "above; the dynamicLRP relevance pass does not run for SigLIP-2 — see "
      "the FINDING cell).")

# Show the source map inline so the locked slice is visually identified.
fig, ax = plt.subplots(1, 1, figsize=(9, 9))
ax.imshow(pil)
ax.set_title(f"locked slice (manifest[-1], D-05)\n{entry['id']}")
ax.set_axis_off()
fig.tight_layout(); plt.show()

In [ ]:
# Cell 5 - dynamicLRP attribution attempt: CAUGHT op-coverage FINDING.
# We call attribute() honestly and CATCH the RuntimeError it raises (the
# `split_with_sizes` coverage gap). This is a recorded per-model RESULT for
# the multi-model dynamic-LRP comparison, NOT a cell error / traceback.
import traceback

LRP_RELEVANCE = None
LRP_FINDING = None
try:
    _res = attribute(model, img_tensor, input_ids, attention_mask)
    LRP_RELEVANCE = _res.relevance
    print("UNEXPECTED: attribute() returned relevance of shape",
          tuple(LRP_RELEVANCE.shape),
          "- the documented SigLIP-2 op-coverage gap did NOT occur. "
          "Investigate before treating this as a finding.")
except RuntimeError as exc:
    LRP_FINDING = repr(exc)
    print("dynamicLRP attribute() raised (EXPECTED, CAUGHT):")
    print(" ", LRP_FINDING)
finally:
    torch.cuda.empty_cache()

## FINDING — dynamicLRP op-coverage on SigLIP-2 (user-accepted; Fallback Ladder DECLINED)

**dynamicLRP op-coverage FINDING.** SigLIP-2-so400m's MAP-pool /
attention-pooling head uses the `split_with_sizes` op
(`SplitWithSizesBackward0` in the autograd graph). The current dynamicLRP
engine registers `SplitWithSizesBackward → SplitBackwardProp` but its Promise
consumer chokes on SigLIP-2's split topology (`'DummyPromise' object is not
iterable` / `No valid curnode candidate was found`), and a 0-dim target form
raises `IndexError`. **`split_with_sizes` is outside the current engine's
covered ops for SigLIP-2 as a contrastive MAP-pool encoder, so no LRP
relevance — and therefore no attribution heatmap — is produced for this
model.** This is exactly the MEDIUM-LOW research risk flagged in `CLAUDE.md`
(*"Whether dynamicLRP covers 100% of SigLIP-2's specific ops out of the box —
MEDIUM-LOW"*).

**This is a recorded per-model RESULT for the multi-model dynamic-LRP
comparison**, the reframed purpose of the project — not a project failure.

**Fallback Ladder: DECLINED by the user** at the 01-03 human-verify checkpoint
(2026-05-19). The user reviewed the smoke-test and the finding and explicitly
declined **every** rung — **no** custom dynamicLRP Promise, **no**
pre-pool / `use_attn_lrp` engineering, **no** vendored LXT, **no** captum
Integrated Gradients baseline. Quote: *"Smoke-test was good, it didn't crash.
Let's leave well enough alone and move on."* This must **not** be
re-attempted later (also recorded in the `attribution.py` module docstring and
`01-03-SUMMARY.md`).

**Phase 1 D-02/D-03 visual-eyeball gate: WAIVED for SigLIP-2.** The three
sanity controls (query-swap, model-randomization, occlusion) are **NOT**
rendered here because **no relevance exists to overlay** — there is nothing to
eyeball. This is a *consciously waived* gate for SigLIP-2 under the reframed
multi-model-comparison purpose, recorded honestly in `01-03-SUMMARY.md`. It is
**not** a silent pass and **not** "all controls passed".

**Smoke-test status — PASS (honest):**

| Item | Result |
|---|---|
| Pinned env + GCS-mirrored SigLIP-2 loads | OK |
| Locked slice (`manifest[-1]`, `"a river"`) `requires_grad` `(1,3,384,384)` tensor built via loaders | OK |
| SigLIP-2 forward + `logits_per_image[0,0]` similarity | OK |
| **Peak forward VRAM measured + recorded** (Cell 4 — STATE.md blocker CLOSED) | OK |
| dynamicLRP coverage probe op count printed (Cell 3) | OK |
| dynamicLRP `attribute()` — `split_with_sizes` coverage gap, **caught** as a recorded finding | FINDING (no heatmap) |
| Three D-02/D-03 sanity controls | **WAIVED for SigLIP-2** (no relevance to render) |
| Notebook runs end-to-end, exit code 0 | OK |